In [ ]:
#instalo STAC
!pip install pystac-client --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 5.0 MB/s eta 0:00:00


In [ ]:
from pystac_client import Client
import requests
# URL actualizada del catálogo STAC para la versión v1
STAC_URL = 'https://earth-search.aws.element84.com/v1'

# Crea un cliente STAC usando la URL del catálogo
client = Client.open(STAC_URL)

bbox_quines = [-66.09895828722148, -32.35715151823955, -65.04436997391068, -31.75468111310571]  # Bounding box [oeste, sur, este, norte]

# Define los parámetros de la búsqueda
search_parameters = {
    "collections": ["sentinel-2-l2a"],  # Colección específica a buscar
    "bbox": bbox_quines,  # Bounding box para Sinop
    "datetime": "2024-12-01T00:00:00Z/2024-12-31T23:59:59Z",  # Intervalo de tiempo (ajusta según necesidad)
    "limit": 1,  # Número de ítems por página de resultados
}

In [ ]:
# Realiza la búsqueda
search = client.search(**search_parameters)

# Obtiene todos los ítems de la búsqueda
items = search.get_all_items()

/usr/local/lib/python3.12/dist-packages/pystac_client/item_search.py:940: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


In [ ]:
#creamos un directorio para guardar las imagenes que bajamos
!mkdir -p data/ejercicio

In [ ]:
# Verifica si se encontraron ítems
if items:
    # Se selecciona el primer ítem de la lista de resultados. Un ítem representa una entidad individual en el catálogo STAC, como una imagen satelital.
    first_item = next(iter(items))

    # Imprime información básica del ítem seleccionado, incluyendo su ID único y la fecha en la que fue capturada la imagen.
    print(f"ID: {first_item.id}")
    print(f"Date: {first_item.datetime}")

    # Define el identificador de la banda o capa que deseas descargar. En este caso, 'visual' podría referirse a una imagen RGB compuesta o similar, dependiendo del catálogo STAC.
    asset_key = 'visual'  # Cambiar según la clave del activo deseado

    # Verifica si el activo deseado ('visual' en este caso) está disponible en los activos del ítem. Los activos representan los recursos disponibles para el ítem, como archivos de imágenes.
    if asset_key in first_item.assets:
        # Accede al activo especificado por 'asset_key' y almacena la referencia en 'asset'.
        asset = first_item.assets[asset_key]

        # Imprime la URL desde donde se puede descargar el activo. Esta URL apunta al archivo de la imagen en sí.
        print(f"Downloading {asset.href}")

        # Realiza una petición GET a la URL del activo para descargar el archivo. Asegúrate de tener los permisos necesarios para acceder a este contenido.
        response = requests.get(asset.href)

        # Abre un archivo en modo de escritura binaria para guardar el contenido descargado. El nombre del archivo se basa en el ID del ítem.
        with open(f"data/ejercicio/TCI.tif", "wb") as file:
            # Escribe el contenido binario de la respuesta (la imagen descargada) en el archivo.
            file.write(response.content)

        # Notifica al usuario que el archivo ha sido descargado con éxito.
        print("Archivo descargado con éxito.")
    else:
        # Si el activo especificado no se encuentra en el ítem, informa al usuario.
        print(f"Asset '{asset_key}' no encontrado en el ítem.")
else:
    # Si no se encontraron ítems que coincidan con los criterios de búsqueda, informa al usuario.
    print("No se encontraron ítems con los criterios de búsqueda proporcionados.")

ID: S2B_19HGE_20241227_0_L2A
Date: 2024-12-27 14:31:45.634000+00:00
Archivo descargado con éxito.


In [ ]:
!ls data/ejercicio

TCI.tif


In [ ]:
from osgeo import gdal
import json
gdal.Info('./data/ejercicio/TCI.tif', format='json', stats=True)

/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


{'description': './data/ejercicio/TCI.tif',
 'driverShortName': 'GTiff',
 'driverLongName': 'GeoTIFF',
 'files': ['./data/ejercicio/TCI.tif'],
 'size': [10980, 10980],
 'coordinateSystem': {'wkt': 'PROJCRS["WGS 84 / UTM zone 19S",\n    BASEGEOGCRS["WGS 84",\n        DATUM["World Geodetic System 1984",\n            ELLIPSOID["WGS 84",6378137,298.257223563,\n                LENGTHUNIT["metre",1]]],\n        PRIMEM["Greenwich",0,\n            ANGLEUNIT["degree",0.0174532925199433]],\n        ID["EPSG",4326]],\n    CONVERSION["UTM zone 19S",\n        METHOD["Transverse Mercator",\n            ID["EPSG",9807]],\n        PARAMETER["Latitude of natural origin",0,\n            ANGLEUNIT["degree",0.0174532925199433],\n            ID["EPSG",8801]],\n        PARAMETER["Longitude of natural origin",-69,\n            ANGLEUNIT["degree",0.0174532925199433],\n            ID["EPSG",8802]],\n        PARAMETER["Scale factor at natural origin",0.9996,\n            SCALEUNIT["unity",1],\n            ID["E

In [ ]:
!pip install leafmap localtileserver --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.9/632.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.8/276.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 k

In [ ]:
import os
import leafmap

m = leafmap.Map(center=[ -32.05591631567263, -65.57166413056608,], zoom=10, backend="folium")

m.add_raster('./data/ejercicio/TCI.tif', layer_name="Image RGB")
m.add_text(
    "Zona de estudio: Imagen RGB",
    position="topright",
    font_size=18,
    font_color="white",
    shadow=True
)
m

Map(center=[-32.1003845, -66.2961265], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

In [ ]:
#imprimo todas las bandas disponibles
first_item.assets

{'aot': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/AOT.tif>,
 'blue': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/B02.tif>,
 'coastal': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/B01.tif>,
 'granule_metadata': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/granule_metadata.xml>,
 'green': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/B03.tif>,
 'nir': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/B08.tif>,
 'nir08': <Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_202

In [ ]:
first_item.assets['nir08']

<Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/B8A.tif>

In [ ]:
first_item.assets['nir08']

<Asset href=https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/19/H/GE/2024/12/S2B_19HGE_20241227_0_L2A/B8A.tif>

In [ ]:
import os

# Get the href from the 'visual' asset
image_href = first_item.assets['visual'].href

# Extract the filename from the href
image_name = os.path.basename(image_href)

print(f"El nombre de la imagen es: {image_name}")

El nombre de la imagen es: TCI.tif


La fecha de la imagen es: 2024-12-27 14:31:45.634000+00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Define the source path of the downloaded image
source_image_path = './data/ejercicio/TCI.tif'

# Define the destination folder in Google Drive
drive_destination_folder = '/content/drive/MyDrive/colab_downloads'

# Create the destination folder if it doesn't exist
os.makedirs(drive_destination_folder, exist_ok=True)

# Define the full destination path in Google Drive
destination_image_path = os.path.join(drive_destination_folder, 'TCI.tif')

# Copy the file to Google Drive
!cp "{source_image_path}" "{destination_image_path}"

print(f"Imagen 'TCI.tif' descargada y copiada a: {destination_image_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imagen 'TCI.tif' descargada y copiada a: /content/drive/MyDrive/colab_downloads/TCI.tif


In [ ]:
from google.colab import drive, files
import os

# Ensure Google Drive is mounted
drive.mount('/content/drive', force_remount=True)

# Define the path to the image in Google Drive (where it was copied previously)
drive_image_path = '/content/drive/MyDrive/colab_downloads/TCI.tif'

# Define a temporary path in the Colab environment to copy the file to
local_temp_path = '/tmp/TCI_from_drive.tif'

# Check if the file exists in Google Drive before copying
if os.path.exists(drive_image_path):
    # Copy the file from Google Drive to the Colab environment
    !cp "{drive_image_path}" "{local_temp_path}"
    print(f"Imagen copiada de Google Drive a {local_temp_path}")

    # Download the file from the Colab environment to your local computer
    files.download(local_temp_path)
    print(f"'{local_temp_path}' ha sido descargado a tu computadora.")
else:
    print(f"Error: El archivo '{drive_image_path}' no se encontró en Google Drive.")

Mounted at /content/drive
Imagen copiada de Google Drive a /tmp/TCI_from_drive.tif


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/tmp/TCI_from_drive.tif' ha sido descargado a tu computadora.


In [ ]:
print(f"Fecha de la imagen: {first_item.datetime}")

Fecha de la imagen: 2024-12-27 14:31:45.634000+00:00
